# LC0-Inspired Chess Engine
## Hybrid RL + Stockfish Training with AlphaZero-style MCTS

This notebook implements a neural network chess engine inspired by Leela Chess Zero (LC0), trained via:
1. **Self-play** with MCTS (Reinforcement Learning)
2. **Supervised play** against Stockfish at configurable skill levels

Architecture: CNN + Residual blocks → Policy head + Value head

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install python-chess --quiet
!apt-get install -y stockfish --quiet

# Verify stockfish installation
import subprocess
result = subprocess.run(['which', 'stockfish'], capture_output=True, text=True)
if result.returncode != 0:
    # Download stockfish binary if apt-get failed
    !wget -q https://github.com/official-stockfish/Stockfish/releases/download/sf_16/stockfish-ubuntu-x86-64-avx2.tar
    !tar -xf stockfish-ubuntu-x86-64-avx2.tar
    !mv stockfish/stockfish-ubuntu-x86-64-avx2 /usr/local/bin/stockfish
    !chmod +x /usr/local/bin/stockfish
    print('Stockfish downloaded and installed.')
else:
    print(f'Stockfish found at: {result.stdout.strip()}')

# Final check
result2 = subprocess.run(['stockfish'], input='quit\n', capture_output=True, text=True, timeout=5)
print('Stockfish OK:', 'Stockfish' in result2.stdout or result2.returncode == 0)

## Cell 2 — Imports

In [ ]:
import os
import sys
import time
import math
import random
import copy
import glob
import collections
import datetime
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import chess
import chess.pgn
import chess.engine

# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 3 — Configuration

In [ ]:
# ============================================================
#  CONFIGURATION — Edit these values to tune training
# ============================================================

# --- Quick test mode: set True to run fast smoke-test ---
QUICK_TEST_MODE = False

# --- Model architecture ---
NUM_RESIDUAL_BLOCKS = 6       # Number of residual blocks (6-10 recommended)
NUM_FILTERS        = 128      # Convolutional filters per block

# --- Training hyperparameters ---
LEARNING_RATE   = 1e-3
BATCH_SIZE      = 128
WEIGHT_DECAY    = 1e-4
POLICY_LOSS_WEIGHT = 1.0
VALUE_LOSS_WEIGHT  = 1.0

# --- MCTS ---
MCTS_SIMULATIONS      = 50    # Simulations per move (50-200)
MCTS_C_PUCT           = 1.5   # Exploration constant
DIRICHLET_ALPHA       = 0.3   # Dirichlet noise alpha
DIRICHLET_EPSILON     = 0.25  # Fraction of Dirichlet noise mixed in
TEMPERATURE_MOVES     = 30    # Moves before switching to greedy play
TEMPERATURE_INIT      = 1.0

# --- Self-play & training loop ---
SELF_PLAY_GAMES_PER_ITER   = 4    # Self-play games per training iteration
STOCKFISH_GAMES_PER_ITER   = 2    # Stockfish games per training iteration
TRAIN_STEPS_PER_ITER       = 50   # Gradient steps per iteration
TOTAL_ITERATIONS           = 20   # Total training iterations
REPLAY_BUFFER_SIZE         = 50000

# --- Stockfish ---
STOCKFISH_PATH  = 'stockfish'
STOCKFISH_SKILLS = [0, 2, 5]      # Skill levels to cycle through
STOCKFISH_TIME   = 0.05           # Time per move in seconds

# --- Checkpointing ---
CHECKPOINT_INTERVAL_MIN = 15      # Save checkpoint every N minutes
MAX_CHECKPOINTS         = 5       # Keep last N checkpoints
USE_GOOGLE_DRIVE        = True    # Mount Google Drive for checkpoints

# --- Max game length ---
MAX_GAME_MOVES = 200

# Override for quick test mode
if QUICK_TEST_MODE:
    print('*** QUICK TEST MODE ENABLED ***')
    NUM_RESIDUAL_BLOCKS      = 2
    NUM_FILTERS              = 32
    MCTS_SIMULATIONS         = 10
    SELF_PLAY_GAMES_PER_ITER = 2
    STOCKFISH_GAMES_PER_ITER = 1
    TRAIN_STEPS_PER_ITER     = 10
    TOTAL_ITERATIONS         = 3
    REPLAY_BUFFER_SIZE       = 5000
    BATCH_SIZE               = 32
    CHECKPOINT_INTERVAL_MIN  = 999  # Disable auto-checkpoint in test

print('Configuration loaded.')

## Cell 4 — Google Drive Mount & Checkpoint Directory

In [ ]:
# Mount Google Drive for persistent checkpoint storage
CHECKPOINT_DIR = '/content/checkpoints'  # fallback

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        CHECKPOINT_DIR = '/content/drive/MyDrive/chess_engine_checkpoints'
        print(f'Google Drive mounted. Checkpoints → {CHECKPOINT_DIR}')
    except Exception as e:
        print(f'Drive mount failed ({e}), using local path.')
        CHECKPOINT_DIR = '/content/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('/content/pgn_games', exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

## Cell 5 — Board Encoding (LC0-style input planes)

In [ ]:
# ============================================================
#  BOARD ENCODING
#  Encodes chess positions as 112-plane 8x8 tensors,
#  following the AlphaZero / LC0 input format.
# ============================================================

NUM_HISTORY_STATES = 8   # Current + 7 previous states
PLANES_PER_STATE   = 13  # 12 piece planes + 1 repetition plane
EXTRA_PLANES       = 8   # Castling(4), side(1), halfmove(1), fullmove(1), zeros(1)
TOTAL_PLANES       = NUM_HISTORY_STATES * PLANES_PER_STATE + EXTRA_PLANES  # 112

# Piece type ordering (indices 0-5 for white, 6-11 for black)
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP,
               chess.ROOK, chess.QUEEN, chess.KING]


def board_to_planes(board: chess.Board, history: list = None) -> np.ndarray:
    """
    Encode a chess.Board (plus optional history) into a (112, 8, 8) float32 array.
    
    Plane layout:
      [0..103] : 8 historical states × 13 planes each
                 per state: planes 0-5 = white pieces (P,N,B,R,Q,K)
                            planes 6-11= black pieces (p,n,b,r,q,k)
                            plane  12  = repetition flag
      [104]    : white queenside castling
      [105]    : white kingside castling
      [106]    : black queenside castling
      [107]    : black kingside castling
      [108]    : side to move (1=black, 0=white)
      [109]    : halfmove clock / 100 (normalised)
      [110]    : fullmove number / 100 (normalised)
      [111]    : all-ones constant plane
    """
    planes = np.zeros((TOTAL_PLANES, 8, 8), dtype=np.float32)

    # Build history list (current board is first, then past states)
    if history is None:
        history = []
    states = [board] + list(reversed(history))  # most recent first
    states = states[:NUM_HISTORY_STATES]          # cap at 8

    for t, state in enumerate(states):
        offset = t * PLANES_PER_STATE
        # Piece planes
        for i, pt in enumerate(PIECE_TYPES):
            for sq in state.pieces(pt, chess.WHITE):
                row, col = divmod(sq, 8)
                planes[offset + i, row, col] = 1.0
            for sq in state.pieces(pt, chess.BLACK):
                row, col = divmod(sq, 8)
                planes[offset + 6 + i, row, col] = 1.0
        # Repetition plane (simplified: 1 if position repeated)
        if state.is_repetition(2):
            planes[offset + 12, :, :] = 1.0

    # Extra planes (indices 104-111)
    base = NUM_HISTORY_STATES * PLANES_PER_STATE  # 104
    if board.has_queenside_castling_rights(chess.WHITE):
        planes[base + 0, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.WHITE):
        planes[base + 1, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK):
        planes[base + 2, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):
        planes[base + 3, :, :] = 1.0
    if board.turn == chess.BLACK:
        planes[base + 4, :, :] = 1.0
    planes[base + 5, :, :] = board.halfmove_clock / 100.0
    planes[base + 6, :, :] = board.fullmove_number / 100.0
    planes[base + 7, :, :] = 1.0  # constant ones plane

    return planes


# ============================================================
#  MOVE ENCODING
#  Encode/decode chess moves as policy indices.
#  We use a flat index over (from_sq, to_sq, promotion) space.
#  Total: 64*64 + 3*8*2 = 4096 + 48 = 4144 (we'll use 4672 like LC0)
# ============================================================

POLICY_SIZE = 4672  # LC0-style policy vector size

# Pre-build move → index mapping
_MOVE_TO_IDX = {}
_IDX_TO_MOVE = {}

def _build_move_index():
    """Build a mapping from UCI move string to policy index."""
    idx = 0
    # Normal moves: from_sq * 64 + to_sq
    for from_sq in range(64):
        for to_sq in range(64):
            move = chess.Move(from_sq, to_sq)
            uci = move.uci()
            _MOVE_TO_IDX[uci] = idx
            _IDX_TO_MOVE[idx] = uci
            idx += 1
    # Promotion moves (under-promotions to N, B, R for both colors)
    for from_sq in range(64):
        for to_sq in range(64):
            for promo in [chess.KNIGHT, chess.BISHOP, chess.ROOK]:
                move = chess.Move(from_sq, to_sq, promotion=promo)
                uci = move.uci()
                _MOVE_TO_IDX[uci] = idx
                _IDX_TO_MOVE[idx] = uci
                idx += 1
    return idx

_ACTUAL_POLICY_SIZE = _build_move_index()
POLICY_SIZE = _ACTUAL_POLICY_SIZE
print(f'Policy index built: {POLICY_SIZE} possible moves')


def move_to_index(move: chess.Move) -> int:
    """Convert a chess.Move to policy index."""
    # Normalise queen promotions to just from→to
    if move.promotion == chess.QUEEN:
        uci = chess.Move(move.from_square, move.to_square).uci()
    else:
        uci = move.uci()
    return _MOVE_TO_IDX.get(uci, 0)


def get_legal_move_mask(board: chess.Board) -> np.ndarray:
    """Return a boolean mask of shape (POLICY_SIZE,) for legal moves."""
    mask = np.zeros(POLICY_SIZE, dtype=np.float32)
    for move in board.legal_moves:
        idx = move_to_index(move)
        if idx < POLICY_SIZE:
            mask[idx] = 1.0
    return mask


print('Board encoding utilities ready.')

## Cell 6 — Neural Network Architecture

In [ ]:
# ============================================================
#  NEURAL NETWORK
#  LC0-inspired CNN with residual blocks.
#  Input:  (batch, 112, 8, 8)
#  Output: policy logits (batch, POLICY_SIZE)
#          value scalar  (batch, 1)
# ============================================================

class ResidualBlock(nn.Module):
    """Standard residual block with two 3×3 convolutions + skip connection."""

    def __init__(self, num_filters: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(num_filters)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.relu(x + residual)   # skip connection
        return x


class ChessNet(nn.Module):
    """
    LC0-inspired chess neural network.

    Architecture:
      1. Initial 3×3 conv to project input planes → num_filters
      2. N residual blocks
      3. Policy head: conv → flatten → linear → POLICY_SIZE logits
      4. Value  head: conv → flatten → linear → tanh → scalar
    """

    def __init__(self, num_filters: int = NUM_FILTERS,
                 num_residual_blocks: int = NUM_RESIDUAL_BLOCKS,
                 policy_size: int = POLICY_SIZE):
        super().__init__()
        self.num_filters = num_filters
        self.policy_size = policy_size

        # --- Input projection ---
        self.input_conv = nn.Conv2d(TOTAL_PLANES, num_filters, 3, padding=1, bias=False)
        self.input_bn   = nn.BatchNorm2d(num_filters)

        # --- Residual tower ---
        self.residual_tower = nn.Sequential(
            *[ResidualBlock(num_filters) for _ in range(num_residual_blocks)]
        )

        # --- Policy head ---
        self.policy_conv = nn.Conv2d(num_filters, 2, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * 8 * 8, policy_size)

        # --- Value head ---
        self.value_conv  = nn.Conv2d(num_filters, 1, 1, bias=False)
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(8 * 8, 256)
        self.value_fc2   = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, TOTAL_PLANES, 8, 8) float tensor
        Returns:
            policy_logits: (batch, POLICY_SIZE)
            value:         (batch, 1)  in [-1, 1]
        """
        # Shared trunk
        x = F.relu(self.input_bn(self.input_conv(x)))
        x = self.residual_tower(x)

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)   # flatten: (batch, 2*8*8)
        policy_logits = self.policy_fc(p)

        # Value head
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)   # flatten: (batch, 8*8)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))  # scalar in [-1, 1]

        return policy_logits, value

    def predict(self, board_planes: np.ndarray, legal_mask: np.ndarray):
        """
        Single-position inference (no grad).
        Returns: policy_probs (np.ndarray), value (float)
        """
        self.eval()
        with torch.no_grad():
            x = torch.FloatTensor(board_planes).unsqueeze(0).to(DEVICE)
            logits, val = self(x)
            logits = logits.squeeze(0).cpu().numpy()  # (POLICY_SIZE,)
            # Mask illegal moves with -inf before softmax
            logits[legal_mask == 0] = -1e9
            exp_logits = np.exp(logits - logits.max())
            policy_probs = exp_logits / (exp_logits.sum() + 1e-8)
            value = val.item()
        return policy_probs, value


# Instantiate the network
model = ChessNet(
    num_filters=NUM_FILTERS,
    num_residual_blocks=NUM_RESIDUAL_BLOCKS,
    policy_size=POLICY_SIZE
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model created: {total_params:,} parameters')
print(f'Input planes: {TOTAL_PLANES}  |  Policy size: {POLICY_SIZE}')

## Cell 7 — MCTS Implementation

In [ ]:
# ============================================================
#  MONTE CARLO TREE SEARCH (AlphaZero-style)
#
#  Key formulas:
#    UCB(s,a) = Q(s,a) + c_puct * P(s,a) * sqrt(N(s)) / (1 + N(s,a))
#    where Q = average value, P = prior from policy net, N = visit counts
# ============================================================

class MCTSNode:
    """A single node in the MCTS tree."""

    __slots__ = ['board', 'parent', 'move', 'children',
                 'visit_count', 'value_sum', 'prior', 'is_expanded']

    def __init__(self, board: chess.Board, parent=None,
                 move: chess.Move = None, prior: float = 0.0):
        self.board       = board.copy()  # snapshot of the board at this node
        self.parent      = parent
        self.move        = move          # move that led to this node
        self.children    = {}            # move → MCTSNode
        self.visit_count = 0
        self.value_sum   = 0.0
        self.prior       = prior         # P(s,a) from policy net
        self.is_expanded = False

    @property
    def q_value(self) -> float:
        """Mean action value Q(s,a)."""
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count

    def ucb_score(self, parent_visit_count: int, c_puct: float) -> float:
        """Upper Confidence Bound for Trees (PUCT variant)."""
        exploration = c_puct * self.prior * math.sqrt(parent_visit_count) / (1 + self.visit_count)
        return self.q_value + exploration

    def best_child(self, c_puct: float) -> 'MCTSNode':
        """Select child with highest UCB score."""
        return max(self.children.values(),
                   key=lambda c: c.ucb_score(self.visit_count, c_puct))


class MCTS:
    """
    AlphaZero-style Monte Carlo Tree Search.

    At each simulation:
      1. SELECT: traverse tree using UCB until a leaf
      2. EXPAND: call neural network to get (policy, value)
      3. BACKPROPAGATE: update visit counts and value sums
    """

    def __init__(self, model: ChessNet, c_puct: float = MCTS_C_PUCT,
                 num_simulations: int = MCTS_SIMULATIONS,
                 dirichlet_alpha: float = DIRICHLET_ALPHA,
                 dirichlet_epsilon: float = DIRICHLET_EPSILON):
        self.model              = model
        self.c_puct             = c_puct
        self.num_simulations    = num_simulations
        self.dirichlet_alpha    = dirichlet_alpha
        self.dirichlet_epsilon  = dirichlet_epsilon

    def run(self, board: chess.Board, history: list = None,
            add_dirichlet: bool = True) -> dict:
        """
        Run MCTS from the given board position.

        Returns:
            A dict {move: visit_count} for all legal moves from root.
        """
        root = MCTSNode(board)
        self._expand(root, history)

        # Add Dirichlet noise to root priors for exploration
        if add_dirichlet and len(root.children) > 0:
            self._add_dirichlet_noise(root)

        for _ in range(self.num_simulations):
            node = root
            search_path = [node]

            # 1. SELECT — walk down the tree
            while node.is_expanded and not node.board.is_game_over():
                if not node.children:
                    break
                node = node.best_child(self.c_puct)
                search_path.append(node)

            # 2. EXPAND — evaluate leaf with neural network
            if node.board.is_game_over():
                value = self._terminal_value(node.board)
            elif not node.is_expanded:
                value = self._expand(node, history)
            else:
                value = node.q_value

            # 3. BACKPROPAGATE — update stats along the path
            self._backpropagate(search_path, value)

        # Return visit count distribution
        return {move: child.visit_count
                for move, child in root.children.items()}

    def _expand(self, node: MCTSNode, history: list = None) -> float:
        """
        Expand a node by calling the neural network.
        Returns the value estimate for the current player.
        """
        planes    = board_to_planes(node.board, history)
        legal_mask = get_legal_move_mask(node.board)
        policy_probs, value = self.model.predict(planes, legal_mask)

        # Create child nodes for all legal moves
        for move in node.board.legal_moves:
            idx   = move_to_index(move)
            prior = float(policy_probs[idx]) if idx < len(policy_probs) else 1e-8
            child_board = node.board.copy()
            child_board.push(move)
            node.children[move] = MCTSNode(
                child_board, parent=node, move=move, prior=prior
            )

        node.is_expanded = True
        return value

    def _backpropagate(self, search_path: list, value: float):
        """Propagate value back up the search path."""
        for node in reversed(search_path):
            node.visit_count += 1
            node.value_sum   += value
            value = -value  # flip value for opponent's perspective

    def _terminal_value(self, board: chess.Board) -> float:
        """Return the game outcome from the perspective of the side that just moved."""
        result = board.result()
        if result == '1-0':
            return 1.0 if board.turn == chess.BLACK else -1.0
        elif result == '0-1':
            return 1.0 if board.turn == chess.WHITE else -1.0
        else:
            return 0.0  # draw

    def _add_dirichlet_noise(self, root: MCTSNode):
        """Add Dirichlet noise to root priors to encourage exploration."""
        n = len(root.children)
        noise = np.random.dirichlet([self.dirichlet_alpha] * n)
        eps   = self.dirichlet_epsilon
        for child, eta in zip(root.children.values(), noise):
            child.prior = (1 - eps) * child.prior + eps * eta

    def get_policy_target(self, visit_counts: dict, temperature: float = 1.0) -> np.ndarray:
        """
        Convert MCTS visit counts to a policy target vector.
        temperature=1 → proportional to visits; temperature→0 → argmax.
        """
        target = np.zeros(POLICY_SIZE, dtype=np.float32)
        if not visit_counts:
            return target

        moves   = list(visit_counts.keys())
        counts  = np.array([visit_counts[m] for m in moves], dtype=np.float32)

        if temperature == 0 or temperature < 1e-6:
            # Greedy: put all mass on the most visited move
            best_idx = np.argmax(counts)
            counts   = np.zeros_like(counts)
            counts[best_idx] = 1.0
        else:
            counts = counts ** (1.0 / temperature)
            counts = counts / counts.sum()

        for move, prob in zip(moves, counts):
            idx = move_to_index(move)
            if idx < POLICY_SIZE:
                target[idx] = prob

        return target

    def select_move(self, visit_counts: dict, move_number: int) -> chess.Move:
        """
        Sample a move from the visit count distribution.
        Uses temperature sampling for first TEMPERATURE_MOVES moves,
        then switches to greedy.
        """
        if not visit_counts:
            return None

        temp = TEMPERATURE_INIT if move_number < TEMPERATURE_MOVES else 0.0
        moves  = list(visit_counts.keys())
        counts = np.array([visit_counts[m] for m in moves], dtype=np.float32)

        if temp < 1e-6:
            return moves[np.argmax(counts)]

        counts = counts ** (1.0 / temp)
        probs  = counts / counts.sum()
        return np.random.choice(moves, p=probs)


print('MCTS implementation ready.')

## Cell 8 — Replay Buffer & Dataset

In [ ]:
# ============================================================
#  REPLAY BUFFER
#  Stores (state_planes, policy_target, value_target) tuples.
#  Supports random sampling for minibatch training.
# ============================================================

class ReplayBuffer:
    """Ring-buffer for experience replay."""

    def __init__(self, max_size: int = REPLAY_BUFFER_SIZE):
        self.max_size = max_size
        self.buffer   = collections.deque(maxlen=max_size)

    def add(self, state: np.ndarray, policy: np.ndarray, value: float):
        self.buffer.append((state, policy, value))

    def add_game(self, game_data: list):
        """Add a list of (state, policy, value) tuples from a completed game."""
        for item in game_data:
            self.add(*item)

    def sample(self, batch_size: int):
        """Sample a random minibatch."""
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        states, policies, values = zip(*batch)
        return (
            np.stack(states),
            np.stack(policies),
            np.array(values, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)


# Instantiate global buffer
replay_buffer = ReplayBuffer(max_size=REPLAY_BUFFER_SIZE)
print('Replay buffer ready.')

## Cell 9 — Self-Play Game Generation

In [ ]:
# ============================================================
#  SELF-PLAY
#  Both sides use the same neural network + MCTS.
#  Data: (state, mcts_policy, game_outcome) for each position.
# ============================================================

def play_self_play_game(model: ChessNet, mcts: MCTS) -> tuple:
    """
    Play one full self-play game using MCTS.

    Returns:
        game_data : list of (state_planes, policy_target, value_target)
        pgn_str   : PGN string for the game
    """
    board       = chess.Board()
    history     = []           # board history for input encoding
    game_data   = []           # will be filled after game ends
    positions   = []           # (planes, policy_target)
    move_number = 0

    # Build PGN game object
    pgn_game   = chess.pgn.Game()
    pgn_game.headers['Event'] = 'Self-Play'
    pgn_game.headers['White'] = 'LC0-clone'
    pgn_game.headers['Black'] = 'LC0-clone'
    pgn_node = pgn_game

    while not board.is_game_over() and move_number < MAX_GAME_MOVES:
        planes     = board_to_planes(board, history)
        visit_counts = mcts.run(board, history, add_dirichlet=True)

        if not visit_counts:
            break

        # Build policy target from MCTS visit counts
        temp          = TEMPERATURE_INIT if move_number < TEMPERATURE_MOVES else 0.0
        policy_target = mcts.get_policy_target(visit_counts, temperature=max(temp, 0.01))

        # Select move to play
        move = mcts.select_move(visit_counts, move_number)
        if move is None or move not in board.legal_moves:
            # Fallback to random legal move
            move = random.choice(list(board.legal_moves))

        positions.append((planes, policy_target, board.turn))
        history.append(board.copy())
        history = history[-7:]  # keep last 7 states

        pgn_node = pgn_node.add_variation(move)
        board.push(move)
        move_number += 1

    # Determine outcome
    result  = board.result()
    outcome = _result_to_value(result)

    pgn_game.headers['Result'] = result

    # Assign value targets from each side's perspective
    for planes, policy_target, turn in positions:
        # outcome is from White's perspective (+1 White wins, -1 Black wins)
        value_target = outcome if turn == chess.WHITE else -outcome
        game_data.append((planes, policy_target, value_target))

    pgn_str = str(pgn_game)
    return game_data, pgn_str, result


def _result_to_value(result: str) -> float:
    """Map PGN result string to a float in {-1, 0, 1}."""
    if result == '1-0':
        return 1.0
    elif result == '0-1':
        return -1.0
    else:
        return 0.0


print('Self-play utilities ready.')

## Cell 10 — Stockfish Interface & Games

In [ ]:
# ============================================================
#  STOCKFISH INTERFACE
#  Plays games between our engine (using MCTS) and Stockfish.
#  Collects training data and evaluates win rate.
# ============================================================

def find_stockfish() -> str:
    """Locate the Stockfish binary."""
    candidates = [
        '/usr/games/stockfish',
        '/usr/bin/stockfish',
        '/usr/local/bin/stockfish',
        'stockfish',
    ]
    for path in candidates:
        try:
            result = subprocess.run([path], input='quit\n',
                                    capture_output=True, text=True, timeout=3)
            if result.returncode == 0 or 'Stockfish' in result.stdout:
                return path
        except (FileNotFoundError, subprocess.TimeoutExpired):
            continue
    return 'stockfish'  # hope it's on PATH

STOCKFISH_BINARY = find_stockfish()
print(f'Stockfish binary: {STOCKFISH_BINARY}')


def play_vs_stockfish(model: ChessNet, mcts: MCTS,
                      skill_level: int = 5,
                      our_color: chess.Color = chess.WHITE) -> tuple:
    """
    Play one game: our engine vs Stockfish.

    Args:
        skill_level : Stockfish skill level 0-20
        our_color   : Which side our engine plays

    Returns:
        game_data : list of (state_planes, policy_target, value_target)
        pgn_str   : PGN string
        result    : '1-0', '0-1', or '1/2-1/2'
        we_won    : bool (did our engine win?)
    """
    board       = chess.Board()
    history     = []
    game_data   = []
    positions   = []
    move_number = 0

    # PGN setup
    pgn_game = chess.pgn.Game()
    pgn_game.headers['Event'] = f'vs Stockfish (skill {skill_level})'
    white_name = 'LC0-clone' if our_color == chess.WHITE else f'Stockfish-{skill_level}'
    black_name = f'Stockfish-{skill_level}' if our_color == chess.WHITE else 'LC0-clone'
    pgn_game.headers['White'] = white_name
    pgn_game.headers['Black'] = black_name
    pgn_node = pgn_game

    try:
        with chess.engine.SimpleEngine.popen_uci(STOCKFISH_BINARY) as sf:
            sf.configure({'Skill Level': skill_level})
            limit = chess.engine.Limit(time=STOCKFISH_TIME)

            while not board.is_game_over() and move_number < MAX_GAME_MOVES:
                if board.turn == our_color:
                    # --- Our engine's turn ---
                    planes     = board_to_planes(board, history)
                    legal_mask = get_legal_move_mask(board)

                    # Use MCTS for move selection
                    visit_counts  = mcts.run(board, history, add_dirichlet=False)
                    if not visit_counts:
                        break
                    policy_target = mcts.get_policy_target(visit_counts, temperature=0.0)
                    move          = mcts.select_move(visit_counts, move_number)
                    if move is None or move not in board.legal_moves:
                        move = random.choice(list(board.legal_moves))

                    positions.append((planes, policy_target, board.turn))
                else:
                    # --- Stockfish's turn ---
                    result_sf = sf.play(board, limit)
                    move = result_sf.move

                history.append(board.copy())
                history = history[-7:]
                pgn_node = pgn_node.add_variation(move)
                board.push(move)
                move_number += 1

    except Exception as e:
        print(f'  [Stockfish error] {e}')
        # Return whatever we have if engine crashed
        pass

    result  = board.result() if board.is_game_over() else '*'
    outcome = _result_to_value(result)

    # Did our engine win?
    if our_color == chess.WHITE:
        we_won = (result == '1-0')
    else:
        we_won = (result == '0-1')

    pgn_game.headers['Result'] = result

    # Assign value targets
    for planes, policy_target, turn in positions:
        value_target = outcome if turn == chess.WHITE else -outcome
        game_data.append((planes, policy_target, value_target))

    return game_data, str(pgn_game), result, we_won


print('Stockfish interface ready.')

## Cell 11 — Training Step

In [ ]:
# ============================================================
#  TRAINING
#  Samples from replay buffer and updates the network.
#  Loss = policy_loss (cross-entropy) + value_loss (MSE)
# ============================================================

optimizer = optim.Adam(model.parameters(),
                        lr=LEARNING_RATE,
                        weight_decay=WEIGHT_DECAY)

# Learning rate scheduler (cosine annealing)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_ITERATIONS * TRAIN_STEPS_PER_ITER, eta_min=1e-5
)


def train_step(model: ChessNet, optimizer, buffer: ReplayBuffer,
               batch_size: int = BATCH_SIZE) -> dict:
    """
    Perform one gradient update step.

    Returns dict with loss values.
    """
    if len(buffer) < batch_size:
        return None  # not enough data yet

    model.train()
    states, policies, values = buffer.sample(batch_size)

    # Convert to tensors
    states_t   = torch.FloatTensor(states).to(DEVICE)
    policies_t = torch.FloatTensor(policies).to(DEVICE)
    values_t   = torch.FloatTensor(values).unsqueeze(1).to(DEVICE)

    # Forward pass
    policy_logits, value_preds = model(states_t)

    # Policy loss: cross-entropy with soft targets
    log_probs   = F.log_softmax(policy_logits, dim=1)
    policy_loss = -(policies_t * log_probs).sum(dim=1).mean()

    # Value loss: MSE
    value_loss = F.mse_loss(value_preds, values_t)

    # Total loss
    total_loss = (POLICY_LOSS_WEIGHT * policy_loss +
                  VALUE_LOSS_WEIGHT  * value_loss)

    # Backpropagation
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    return {
        'total': total_loss.item(),
        'policy': policy_loss.item(),
        'value': value_loss.item(),
    }


def train_n_steps(model: ChessNet, optimizer, buffer: ReplayBuffer,
                  n_steps: int) -> dict:
    """
    Run n_steps training steps and return averaged metrics.
    """
    metrics = collections.defaultdict(list)
    for _ in range(n_steps):
        result = train_step(model, optimizer, buffer, BATCH_SIZE)
        if result:
            for k, v in result.items():
                metrics[k].append(v)

    if not metrics:
        return {'total': 0.0, 'policy': 0.0, 'value': 0.0}

    return {k: np.mean(v) for k, v in metrics.items()}


print('Training utilities ready.')

## Cell 12 — Checkpointing Utilities

In [ ]:
# ============================================================
#  CHECKPOINTING
#  Saves model + optimizer state with timestamp.
#  Keeps only the last MAX_CHECKPOINTS files.
# ============================================================

_last_checkpoint_time = time.time()


def save_checkpoint(model: ChessNet, optimizer, iteration: int,
                    metrics: dict = None, force: bool = False):
    """Save checkpoint if enough time has passed, or if forced."""
    global _last_checkpoint_time

    elapsed_min = (time.time() - _last_checkpoint_time) / 60.0
    if not force and elapsed_min < CHECKPOINT_INTERVAL_MIN:
        return False

    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    filename  = f'chess_engine_iter{iteration:04d}_{timestamp}.pt'
    filepath  = os.path.join(CHECKPOINT_DIR, filename)

    checkpoint = {
        'iteration':   iteration,
        'timestamp':   timestamp,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'metrics':     metrics or {},
        'config': {
            'num_filters':         NUM_FILTERS,
            'num_residual_blocks': NUM_RESIDUAL_BLOCKS,
            'policy_size':         POLICY_SIZE,
        }
    }

    torch.save(checkpoint, filepath)
    print(f'  [Checkpoint saved] {filepath}')

    # Prune old checkpoints
    _prune_checkpoints()

    _last_checkpoint_time = time.time()
    return True


def _prune_checkpoints():
    """Keep only the last MAX_CHECKPOINTS checkpoint files."""
    ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'chess_engine_*.pt')))
    while len(ckpts) > MAX_CHECKPOINTS:
        os.remove(ckpts.pop(0))


def load_latest_checkpoint(model: ChessNet, optimizer) -> int:
    """
    Load the most recent checkpoint if one exists.
    Returns the iteration number (0 if no checkpoint found).
    """
    ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'chess_engine_*.pt')))
    if not ckpts:
        print('No checkpoint found. Starting fresh.')
        return 0

    latest = ckpts[-1]
    print(f'Loading checkpoint: {latest}')
    ckpt = torch.load(latest, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optim_state'])
    print(f'  Resumed from iteration {ckpt["iteration"]}')
    return ckpt['iteration']


def save_pgn(pgn_str: str, filename: str):
    """Save a PGN game string to file."""
    path = os.path.join('/content/pgn_games', filename)
    with open(path, 'w') as f:
        f.write(pgn_str)


print('Checkpointing utilities ready.')

## Cell 13 — Logging Utilities

In [ ]:
# ============================================================
#  LOGGING
#  Clean progress output for training loop.
# ============================================================

class TrainingLogger:
    """Tracks and displays training metrics."""

    def __init__(self):
        self.history = []  # list of per-iteration dicts

    def log_iteration(self, iteration: int, metrics: dict):
        metrics['iteration'] = iteration
        metrics['time']      = datetime.datetime.now().strftime('%H:%M:%S')
        self.history.append(metrics)
        self._print(metrics)

    def _print(self, m: dict):
        sep = '─' * 65
        print(f'\n{sep}')
        print(f'  Iteration {m["iteration"]:4d}  |  {m["time"]}')
        print(f'{sep}')
        print(f'  Loss  total={m.get("loss_total", 0):.4f}  '
              f'policy={m.get("loss_policy", 0):.4f}  '
              f'value={m.get("loss_value", 0):.4f}')
        print(f'  Self-play games:  {m.get("sp_games", 0):3d}  '
              f'| positions added: {m.get("sp_positions", 0):5d}')
        print(f'  Stockfish games:  {m.get("sf_games", 0):3d}  '
              f'| wins: {m.get("sf_wins", 0):3d}  '
              f'| win rate: {m.get("sf_win_rate", 0)*100:.1f}%')
        print(f'  Avg game length:  {m.get("avg_game_len", 0):.1f} moves')
        print(f'  Buffer size:      {m.get("buffer_size", 0):6d}')
        print(f'  LR:               {m.get("lr", 0):.2e}')
        print(sep)


logger = TrainingLogger()
print('Logger ready.')

## Cell 14 — Resume or Start Fresh

In [ ]:
# ============================================================
#  OPTIONALLY RESUME FROM CHECKPOINT
# ============================================================

START_ITERATION = load_latest_checkpoint(model, optimizer)
print(f'Will train from iteration {START_ITERATION} to {START_ITERATION + TOTAL_ITERATIONS}')

## Cell 15 — Main Training Loop

In [ ]:
# ============================================================
#  MAIN TRAINING LOOP
#
#  Each iteration:
#  1. Self-play games (MCTS)     → fill replay buffer
#  2. Stockfish games            → fill replay buffer + track win rate
#  3. Train for N steps          → update network weights
#  4. Log metrics
#  5. Save checkpoint if needed
#  6. Save last PGN files
# ============================================================

mcts = MCTS(
    model=model,
    c_puct=MCTS_C_PUCT,
    num_simulations=MCTS_SIMULATIONS,
    dirichlet_alpha=DIRICHLET_ALPHA,
    dirichlet_epsilon=DIRICHLET_EPSILON,
)

# Warm up buffer with a few random games before training starts
print('Warming up replay buffer with random games...')
_dummy_board = chess.Board()
_dummy_legal = list(_dummy_board.legal_moves)
for _ in range(min(50, BATCH_SIZE)):  # add some random noise
    planes = board_to_planes(_dummy_board)
    policy = np.zeros(POLICY_SIZE, dtype=np.float32)
    idx    = move_to_index(random.choice(_dummy_legal))
    if idx < POLICY_SIZE:
        policy[idx] = 1.0
    replay_buffer.add(planes, policy, 0.0)
print(f'Buffer pre-filled with {len(replay_buffer)} entries.\n')

# ─── Training Loop ───────────────────────────────────────────
stockfish_skill_cycle = 0  # cycle through STOCKFISH_SKILLS

for iteration in range(START_ITERATION, START_ITERATION + TOTAL_ITERATIONS):
    iter_start = time.time()
    print(f'\n>>> Starting iteration {iteration} <<<')

    sp_games      = 0
    sp_positions  = 0
    game_lengths  = []
    last_sp_pgn   = ''

    # ── 1. Self-Play ──────────────────────────────────────────
    print(f'  Self-play: {SELF_PLAY_GAMES_PER_ITER} games...')
    for g in range(SELF_PLAY_GAMES_PER_ITER):
        try:
            game_data, pgn_str, result = play_self_play_game(model, mcts)
            replay_buffer.add_game(game_data)
            sp_games     += 1
            sp_positions += len(game_data)
            game_lengths.append(len(game_data))
            last_sp_pgn   = pgn_str
            print(f'    SP game {g+1}/{SELF_PLAY_GAMES_PER_ITER}: {len(game_data)} positions | result={result}')
        except Exception as e:
            print(f'    SP game {g+1} failed: {e}')

    # ── 2. Stockfish Games ────────────────────────────────────
    sf_wins      = 0
    sf_games     = 0
    last_sf_pgn  = ''
    skill = STOCKFISH_SKILLS[stockfish_skill_cycle % len(STOCKFISH_SKILLS)]
    stockfish_skill_cycle += 1

    print(f'  Stockfish games ({STOCKFISH_GAMES_PER_ITER} games, skill={skill})...')
    for g in range(STOCKFISH_GAMES_PER_ITER):
        our_color = chess.WHITE if g % 2 == 0 else chess.BLACK
        try:
            game_data, pgn_str, result, won = play_vs_stockfish(
                model, mcts, skill_level=skill, our_color=our_color
            )
            replay_buffer.add_game(game_data)
            sf_games     += 1
            sp_positions += len(game_data)
            game_lengths.append(len(game_data))
            if won:
                sf_wins += 1
            last_sf_pgn = pgn_str
            color_str = 'White' if our_color == chess.WHITE else 'Black'
            won_str   = 'WON' if won else 'lost'
            print(f'    SF game {g+1}/{STOCKFISH_GAMES_PER_ITER} '
                  f'(us={color_str}): result={result} | {won_str}')
        except Exception as e:
            print(f'    SF game {g+1} failed: {e}')

    # ── 3. Train ──────────────────────────────────────────────
    print(f'  Training for {TRAIN_STEPS_PER_ITER} steps...')
    loss_metrics = train_n_steps(model, optimizer, replay_buffer,
                                  TRAIN_STEPS_PER_ITER)

    # ── 4. Log ────────────────────────────────────────────────
    sf_win_rate = sf_wins / sf_games if sf_games > 0 else 0.0
    avg_len     = np.mean(game_lengths) if game_lengths else 0.0
    cur_lr      = optimizer.param_groups[0]['lr']

    iter_metrics = {
        'loss_total':  loss_metrics['total'],
        'loss_policy': loss_metrics['policy'],
        'loss_value':  loss_metrics['value'],
        'sp_games':    sp_games,
        'sp_positions':sp_positions,
        'sf_games':    sf_games,
        'sf_wins':     sf_wins,
        'sf_win_rate': sf_win_rate,
        'avg_game_len':avg_len,
        'buffer_size': len(replay_buffer),
        'lr':          cur_lr,
    }
    logger.log_iteration(iteration, iter_metrics)

    # ── 5. Save PGNs ──────────────────────────────────────────
    if last_sp_pgn:
        save_pgn(last_sp_pgn, f'selfplay_iter{iteration:04d}.pgn')
        print(f'  PGN saved: selfplay_iter{iteration:04d}.pgn')
    if last_sf_pgn:
        save_pgn(last_sf_pgn, f'stockfish_iter{iteration:04d}.pgn')
        print(f'  PGN saved: stockfish_iter{iteration:04d}.pgn')

    # ── 6. Checkpoint ─────────────────────────────────────────
    saved = save_checkpoint(model, optimizer, iteration, iter_metrics)

    elapsed = time.time() - iter_start
    print(f'  Iteration done in {elapsed:.1f}s')


# ─── Final checkpoint ─────────────────────────────────────────
save_checkpoint(model, optimizer,
                START_ITERATION + TOTAL_ITERATIONS,
                force=True)

print('\n' + '='*65)
print('  TRAINING COMPLETE')
print('='*65)

## Cell 16 — Summary & Training Curves

In [ ]:
# ============================================================
#  TRAINING SUMMARY
# ============================================================

import matplotlib.pyplot as plt

if logger.history:
    iters        = [m['iteration'] for m in logger.history]
    loss_total   = [m['loss_total']  for m in logger.history]
    loss_policy  = [m['loss_policy'] for m in logger.history]
    loss_value   = [m['loss_value']  for m in logger.history]
    win_rates    = [m['sf_win_rate'] * 100 for m in logger.history]
    avg_lens     = [m['avg_game_len'] for m in logger.history]

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle('LC0-Clone Training Summary', fontsize=14, fontweight='bold')

    # Loss curves
    axes[0,0].plot(iters, loss_total,  label='Total',  color='black')
    axes[0,0].plot(iters, loss_policy, label='Policy', color='steelblue')
    axes[0,0].plot(iters, loss_value,  label='Value',  color='coral')
    axes[0,0].set_title('Training Loss')
    axes[0,0].set_xlabel('Iteration')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

    # Win rate
    axes[0,1].plot(iters, win_rates, color='green', marker='o')
    axes[0,1].set_title('Win Rate vs Stockfish (%)')
    axes[0,1].set_xlabel('Iteration')
    axes[0,1].set_ylabel('Win Rate %')
    axes[0,1].set_ylim(0, 105)
    axes[0,1].grid(True, alpha=0.3)

    # Average game length
    axes[1,0].plot(iters, avg_lens, color='purple', marker='s')
    axes[1,0].set_title('Average Game Length (moves)')
    axes[1,0].set_xlabel('Iteration')
    axes[1,0].set_ylabel('Moves')
    axes[1,0].grid(True, alpha=0.3)

    # Buffer size
    buf_sizes = [m['buffer_size'] for m in logger.history]
    axes[1,1].plot(iters, buf_sizes, color='darkorange', marker='^')
    axes[1,1].set_title('Replay Buffer Size')
    axes[1,1].set_xlabel('Iteration')
    axes[1,1].set_ylabel('Positions')
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Training curves saved to /content/training_curves.png')
else:
    print('No training history to plot.')

## Cell 17 — Interactive: Play Against the Engine

In [ ]:
# ============================================================
#  PLAY AGAINST THE TRAINED ENGINE
#  Enter your move in UCI format (e.g. 'e2e4', 'd7d5', 'g1f3')
#  Type 'quit' to end the game.
# ============================================================

def display_board(board: chess.Board):
    """Pretty-print the board with column labels."""
    print()
    print('  a b c d e f g h')
    print(' ┌─────────────────┐')
    rows = str(board).split('\n')
    for i, row in enumerate(rows):
        rank = 8 - i
        print(f'{rank}│ {row} │{rank}')
    print(' └─────────────────┘')
    print('  a b c d e f g h')
    print(f'  Turn: {"White" if board.turn == chess.WHITE else "Black"}')
    print()


def engine_move(board: chess.Board, history: list) -> chess.Move:
    """Ask the engine for its best move."""
    mcts_play = MCTS(model=model, num_simulations=MCTS_SIMULATIONS,
                     c_puct=MCTS_C_PUCT, dirichlet_epsilon=0.0)  # no Dirichlet at test time
    visit_counts = mcts_play.run(board, history, add_dirichlet=False)
    if not visit_counts:
        return random.choice(list(board.legal_moves))
    return mcts_play.select_move(visit_counts, move_number=999)  # greedy


def play_human_vs_engine(human_color: chess.Color = chess.WHITE):
    """Interactive human vs engine game."""
    board   = chess.Board()
    history = []
    print(f'\nYou are playing as {"White" if human_color == chess.WHITE else "Black"}.')
    print('Enter moves in UCI format (e.g. e2e4). Type "quit" to exit.\n')

    while not board.is_game_over():
        display_board(board)

        if board.turn == human_color:
            # Human's turn
            legal_ucis = [m.uci() for m in board.legal_moves]
            while True:
                raw = input('Your move: ').strip().lower()
                if raw == 'quit':
                    print('Game aborted.')
                    return
                try:
                    move = chess.Move.from_uci(raw)
                    if move in board.legal_moves:
                        break
                    else:
                        print(f'  Illegal move. Legal moves: {" ".join(legal_ucis[:10])}...')
                except ValueError:
                    print('  Invalid UCI format. Example: e2e4')
        else:
            # Engine's turn
            print('Engine thinking...')
            move = engine_move(board, history)
            print(f'Engine plays: {move.uci()}')

        history.append(board.copy())
        history = history[-7:]
        board.push(move)

    display_board(board)
    print(f'Game over! Result: {board.result()}')


# Uncomment the line below to play:
# play_human_vs_engine(human_color=chess.WHITE)
print('Interactive play function ready.')
print('To play, run: play_human_vs_engine(human_color=chess.WHITE)')

## Cell 18 — Export Final Model

In [ ]:
# ============================================================
#  EXPORT FINAL MODEL
#  Saves to Google Drive and also offers a local download.
# ============================================================

final_path = os.path.join(CHECKPOINT_DIR, 'chess_engine_FINAL.pt')
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'num_filters':         NUM_FILTERS,
        'num_residual_blocks': NUM_RESIDUAL_BLOCKS,
        'policy_size':         POLICY_SIZE,
        'total_planes':        TOTAL_PLANES,
    },
    'policy_size': POLICY_SIZE,
}, final_path)
print(f'Final model saved to: {final_path}')

# Also copy to /content for easy download
import shutil
shutil.copy(final_path, '/content/chess_engine_FINAL.pt')
print('Also copied to /content/chess_engine_FINAL.pt')

# Optionally download from Colab
try:
    from google.colab import files
    files.download('/content/chess_engine_FINAL.pt')
    print('Download initiated.')
except Exception:
    print('Not in Colab or download skipped. File is at /content/chess_engine_FINAL.pt')